In [1]:
import plotly.graph_objs as go
import pandas as pd
from plotly.subplots import make_subplots

In [2]:
def load_all_data():

    solcast = pd.read_csv('Data/Solcast/Solcast_Formatted.csv', parse_dates=['DateTime'], index_col='DateTime')
    power_exp = pd.read_csv('Data/Solcast/Power_Expectation.csv', parse_dates=['DateTime'], index_col='DateTime')
    deop = pd.read_csv('Data/DEOP/2023_DEOP.csv', parse_dates=['DateTime'], index_col='DateTime')
    expected = pd.read_csv('Data/DEOP/2023_Expected.csv', parse_dates=['DateTime'], index_col='DateTime')
    
    solcast['P_MaxSun'] = power_exp['P_MaxSolar']
    return solcast, deop, expected

In [3]:
def get_colours():
    colours = [
        '#000000', '#E69F00', '#2CA02C', "#3827D6", 
        '#9467BD', '#8C564B', '#E377C2', '#7F7F7F', 
        "#DB1212", '#17BECF', "#E600C0", '#56B4E9'
    ]
    return colours

In [4]:
import plotly.graph_objects as go

def PlotAll(dfs, y_label, data_labels, title):
    width_cm = 32
    height_cm = 12
    PIXEL_PER_CM = 37.8
    colours = get_colours()
    fig = go.Figure()
    
    for i in range(len(dfs)):
        fig.add_trace(go.Scatter(
            x=dfs[i].index, 
            y=dfs[i], 
            mode='lines', 
            name=data_labels[i], 
            opacity=0.6, 
            line_color=colours[i % len(colours)],
            line=dict(width=3.5) 
        ))
    
    fig.update_layout(
        
        width=width_cm * PIXEL_PER_CM,
        height=height_cm * PIXEL_PER_CM,
        autosize=False,
        
        title=dict(
            text=title,
            font=dict(size=26)
        ),
        xaxis=dict(
            title=dict(text='Time', font=dict(size=1)),
            tickfont=dict(size=18),
            gridcolor='lightgrey'
        ),
        yaxis=dict(
            title=dict(text=y_label, font=dict(size=22)),
            tickfont=dict(size=18),
            gridcolor='lightgrey'
        ),
        legend=dict(
            font=dict(size=16),
            bgcolor='rgba(255,255,255,0.5)'
        ),
        template='plotly_white',
        margin=dict(l=100, r=50, t=100, b=100)
    )
    
    return fig

In [5]:
def get_problem_days(expected_df, actual_df, threshold=0.1):
    
    df = pd.concat([expected_df, actual_df], axis=1)
    daily = df.groupby(pd.Grouper(freq='d')).agg(
        expected=('expected', 'sum'),
        true=('power-gen-pv-ave', 'sum'),
    )
    return daily[(daily['true'] < daily['expected'] * threshold)]

In [6]:
def calculate_daily(deop, expected=None):
    daily = pd.concat([expected, deop['power-gen-pv-ave']], axis=1).resample('D').sum()
    if expected is None:
         return daily
    daily.columns = ['expected', 'actual']

    daily['performance_ratio'] = daily['actual'] / daily['expected']
    daily = daily[daily['expected'] > 0]

    daily['7d_avg'] = daily['performance_ratio'].rolling(window=7).mean()
    return daily

In [7]:
def get_month_order():
    return ['January', 'February', 'March', 'April', 'May', 'June', 
            'July', 'August', 'September', 'October', 'November', 'December']


In [8]:
def get_monthly_matrix(df, column, time):
    months_order = get_month_order()
    df['Month'] = df.index.month_name()
    df[time] = df.index.floor(time).time 
    grouped = df.groupby(['Month', time])[[column]].mean().reindex(months_order, level='Month')
    return grouped[column].unstack(level='Month')

In [9]:
def plot_monthly_box(df, column, title, noZero=False):
    months_order = get_month_order()
    df = calculate_daily(df)
    if noZero:
        df = df[df[column] > 0]
    df['Month'] = df.index.month_name()
    fig = go.Figure()
    fig.add_trace(go.Box(
        x=df['Month'],
        y=df[column],
        name='True Generation'
    ))
    fig.update_layout(
        title=title,
        yaxis_title='Total Daily Generation',
        xaxis=dict(categoryorder='array', categoryarray=months_order)
    )
    return fig

In [10]:
def plot_daily_profilenew(monthly_matrix, group, title):


    PIXEL_PER_CM = 37.8
    width_cm=25.83
    height_cm=13.65
    month_data = [monthly_matrix[m] for m in monthly_matrix.columns]

    fig = PlotAll(
        dfs=month_data,
        y_label='Average Power [kW]',
        data_labels=monthly_matrix.columns,
        title=title
    )

    peak_values = monthly_matrix.max()
    peak_times = monthly_matrix.idxmax()
    annotations = []
    colours = get_colours()
    for month in group:
        if month in peak_times and month in peak_values:
            time = peak_times[month]  
            value = peak_values[month]
            month_idx = list(monthly_matrix.columns).index(month)
            text_colour = colours[month_idx % len(colours)]
            annotations.append(
                dict(
                    x=time, 
                    y=value,
                    text=f"<b>{month[:3]}</b>",  
                    showarrow=False,
                    font=dict(size=18, color=text_colour), 
                    yanchor='bottom', 
                    yshift=5    
                )
            )

    fig.update_layout(width=1200,
        height=720,
        annotations=annotations)
    
    fig.update_layout(
        width=width_cm * PIXEL_PER_CM,
        height=height_cm * PIXEL_PER_CM,
        autosize=False, 
        annotations=annotations
    )
    return fig

In [11]:
import pandas as pd

def plot_daily_profile(monthly_matrix, group, title):
    PIXEL_PER_CM = 37.8
    width_cm = 29.57
    height_cm = 15.1
    
    month_data = [monthly_matrix[m] for m in monthly_matrix.columns]
    fig = PlotAll(
        dfs=month_data,
        y_label='Average Power [kW]',
        data_labels=monthly_matrix.columns,
        title=title
    )

    fig.update_traces(line=dict(width=3.5)) 

    peak_values = monthly_matrix.max()
    peak_times = monthly_matrix.idxmax()
    colours = get_colours()
    
    raw_annos = []
    for month in group:
        if month in peak_times and month in peak_values:
            m_time = peak_times[month]
            current_mins = m_time.hour * 60 + m_time.minute
            month_idx = list(monthly_matrix.columns).index(month)
            raw_annos.append({
                'month': month,
                'x_val': m_time, 
                'mins': current_mins,
                'y': peak_values[month],
                'color': colours[month_idx % len(colours)]
            })

    raw_annos.sort(key=lambda x: x['y'])

    annotations = []
    for anno in raw_annos:
        x_offset = 0
        text_anchor = 'center'
        for placed in annotations:
            x_dist = abs(anno['mins'] - placed['mins_tracker'])
            y_dist = abs(anno['y'] - placed['y'])
            if x_dist < 90 and y_dist < (max(peak_values) * 0.05):
                if anno['mins'] >= placed['mins_tracker']:
                    x_offset = 45 
                    text_anchor = 'left'
                else:
                    x_offset = -45
                    text_anchor = 'right'

        annotations.append(dict(
            x=anno['x_val'],
            y=anno['y'],
            mins_tracker=anno['mins'],
            text=f"<b>{anno['month'][:100]}</b>",
            showarrow=False,
            font=dict(size=18, color=anno['color']),
            xanchor=text_anchor,
            xshift=x_offset, 
            yanchor='middle',
            bgcolor="rgba(255, 255, 255, 0.5)"
        ))


    all_times = monthly_matrix.index.tolist()
    tick_vals = [t for t in all_times if t.hour % 4 == 0 and t.minute == 0]
    tick_text = [f"{t.hour:02d}:00" for t in tick_vals]

    fig.update_layout(
        width=width_cm * PIXEL_PER_CM,
        height=height_cm * PIXEL_PER_CM,
        autosize=False,
        annotations=[{k: v for k, v in a.items() if k != 'mins_tracker'} for a in annotations],

        xaxis=dict(
            tickmode='array',
            tickvals=tick_vals,
            ticktext=tick_text,
            tickfont=dict(size=20),       
            title=dict(
                text="Time of Day", 
                font=dict(size=22)    
            )
        ),

        yaxis=dict(
            tickfont=dict(size=20),  
            title=dict(
                font=dict(size=22)  
            )
        ),

        title=dict(
            font=dict(size=26)
        )
    )
    
    return fig

In [12]:
def get_season(month):
    if month in ['December', 'January', 'February']:
        return 'Winter'
    elif month in ['March', 'April', 'May']:
        return 'Spring'
    elif month in ['June', 'July', 'August']:
        return 'Summer'
    else:
        return 'Autumn'

In [13]:
def remove_problem_days(df, problem_days_df):
    dates_to_remove = pd.to_datetime(problem_days_df.index.date)
    filtered_df = df[~df.index.normalize().isin(dates_to_remove)]
    return filtered_df

In [16]:
def plot_seasonal_profiles(df, column, title):
    df = df.copy()
    monthly_matrix = get_monthly_matrix(df, column, '60min')

    seasons = pd.DataFrame()
    seasons['Spring'] = monthly_matrix[['March', 'April', 'May']].mean(axis=1)
    seasons['Summer'] = monthly_matrix[['June', 'July', 'August']].mean(axis=1)
    seasons['Autumn'] = monthly_matrix[['September', 'October', 'November']].mean(axis=1)
    if ('December' in monthly_matrix.columns and 'January' in monthly_matrix.columns and 'February' in monthly_matrix.columns):
        seasons['Winter'] = monthly_matrix[['December', 'January', 'February']].mean(axis=1)    
    else:
        seasons['Winter'] = monthly_matrix[['December']].mean(axis=1)
        
    order = ['Spring', 'Summer', 'Autumn', 'Winter']
    fig = plot_daily_profile(seasons, order, title)
    
    return fig